In [ ]:
%matplotlib inline

# Exploit an HDF5 cache file

## Problem

You store your discipline cache in a HDF5 file,
and you want to use it elsewhere.
You will know how to retrieve the cached data.

## Solution

HDF5 stores data in nodes within a file to enable different blocks of data to be stored.
For instance, in GEMSEO, it is possible to store different discipline caches
in the same HDF5 file by specifying a different node name for each one (see note below).
The node name is required to access the desired data.

## Step-by-step guide


In [ ]:
from __future__ import annotations

from numpy import array

from gemseo.caches.hdf5 import HDF5Cache
from gemseo.disciplines.analytic import AnalyticDiscipline

### 1. Create a hdf5 cache

Here, we consider different analytic disciplines, using the same HDF5 cache.



In [ ]:
disciplines = [
    AnalyticDiscipline({"y": "2*x+9"}, name="Discipline 0"),
    AnalyticDiscipline({"y": "3*x+1"}, name="Discipline 1"),
]
for discipline in disciplines:
    discipline.set_cache(
        discipline.CacheType.HDF5,
        hdf_file_path="cache_analytic.hdf5",
    )
    discipline.execute({"x": array([2])})
    discipline.execute({"x": array([9])})
    cache = discipline.cache
    print(f"Cache '{cache.name}':\n", cache.to_dataset())

!!! note
    By default, the cache of each discipline is stored in a node
    named using the discipline name.
    Here, a file (*cache_analytic.hdf5*) is created with 2 nodes:
    *Discipline 0* and *Discipline 1*.
    To set another node name, you can use the
    [hdf_node_path][gemseo.caches.hdf5.HDF5Cache.hdf_node_path] argument.

### 2. Retrieve the first cache using the HDF5 file

You can use the HDF5 file to re-create a cache.
To do so, you have to specify the file, and the node:



In [ ]:
new_cache = HDF5Cache(hdf_file_path="cache_analytic.hdf5", hdf_node_path="Discipline 0")
new_cache.to_dataset()

### 3. Use the same cache in another execution

Here, we create a discipline with its default name
(the class name in GEMSEO),
but we set the HDF cache with both the same hdf file
and hdf node than *Discipline 0*.



In [ ]:
new_discipline = AnalyticDiscipline({"y": "2*x+9"})
new_discipline.set_cache(
    discipline.CacheType.HDF5,
    hdf_file_path="cache_analytic.hdf5",
    hdf_node_path="Discipline 0",
)
new_discipline.execute({"x": array([29])})
new_discipline.cache.to_dataset()

## Summary

Data may be stored across multiple HDF5 nodes within a single HDF5 file.
When saving or extracting data, the `hdf_node_path` attribute must be specified.
By default, this node path corresponds to the discipline name.

